# Wire Myography Analysis
**Author:** Agnieszka Karaś, PhD — agaakaras@gmail.com

**Instructions:** Run cells in order (▶️). Upload your LabChart `.txt` file with comments regarding reagent addition.

---

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl', '-q'])
print('Libraries ready')

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from google.colab import files

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 1.1, 'lines.linewidth': 1.6, 'figure.dpi': 130,
})
CHANNEL_COLS = [1, 2, 3, 4, 5, 6, 7, 8]
CH_COLORS = plt.cm.tab10(np.linspace(0, 0.8, 8))

# Load and read txt file with results, read comments
def load_labchart(file_bytes):
    df = pd.read_csv(
        io.BytesIO(file_bytes), sep='\t', decimal=',', skiprows=9,
        header=None, names=['t',1,2,3,4,5,6,7,8,'comment'],
        dtype={c: float for c in ['t']+CHANNEL_COLS}, keep_default_na=False,
    )
    df['comment'] = df['comment'].astype(str).str.strip()
    for c in CHANNEL_COLS:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

def preview_comments(df):
    return df[df['comment'].str.startswith('#*')][['t','comment']].reset_index(drop=True)

def _find_row(df, label):
    search = label.strip().lstrip('#').lstrip('*').strip()
    norm   = df['comment'].str.lstrip('#').str.lstrip('*').str.strip()
    hits   = np.where(norm == search)[0]
    if len(hits) == 0:
        avail = df.loc[df['comment'].str.startswith('#*'), 'comment'].unique()
        raise ValueError(f"Label '{label}' not found.\nAvailable: {list(avail)}")
    return int(hits[0])

# Calculation of contraction and relaxation
def compute_contraction(df, start_label, end_label):
    s = _find_row(df, start_label); e = _find_row(df, end_label)
    return (df.iloc[s:e][CHANNEL_COLS].max() - df.iloc[s-1][CHANNEL_COLS]).rename('mN')

def compute_cumulative_contraction(df, dose_labels, end_label):
    baseline = df.iloc[_find_row(df, dose_labels[0]) - 1][CHANNEL_COLS]
    records = []
    for i, label in enumerate(dose_labels):
        s = _find_row(df, label)
        e = (_find_row(df, dose_labels[i+1]) if i < len(dose_labels)-1
             else _find_row(df, end_label))
        records.append(df.iloc[s:e][CHANNEL_COLS].max() - baseline)
    return pd.DataFrame(records, index=dose_labels)

def compute_cumulative_relaxation(df, dose_labels, end_label):
    baseline = df.iloc[_find_row(df, dose_labels[0]) - 1][CHANNEL_COLS]
    records = []
    for i, label in enumerate(dose_labels):
        s = _find_row(df, label)
        e = (_find_row(df, dose_labels[i+1]) if i < len(dose_labels)-1
             else _find_row(df, end_label))
        records.append(baseline - df.iloc[s:e][CHANNEL_COLS].min())
    return pd.DataFrame(records, index=dose_labels)

# Create table with results
def analyse_experiment(df, phe_mode, phe_labels, ach_labels, snp_labels, l):

    rows_mn  = []
    rows_pct = []

    def add_mn(label, series):
        row = {'metric': label}
        for ch in CHANNEL_COLS:
            row[f'ch{ch}'] = round(float(series[ch]), 4)
        rows_mn.append(row)

    def add_pct(label, series):
        row = {'metric': label}
        for ch in CHANNEL_COLS:
            row[f'ch{ch}'] = round(float(series[ch]), 4)
        rows_pct.append(row)

    # KCl 60 mM
    kcl60 = compute_contraction(df, l['KCl60'], l['KCl60_end'])
    add_mn('KCl60_mN', kcl60)

    # Phenylephrine - dose response or single dose induced contraction
    if phe_mode == 'dose_response':
        phe_abs = compute_cumulative_contraction(df, phe_labels, l['phe_end'])
        phe_pct = phe_abs.div(kcl60, axis=1) * 100
        for label in phe_labels:
            add_mn( f'Phe_{label}_mN',    phe_abs.loc[label])
            add_pct(f'Phe_{label}_%_KCl', phe_pct.loc[label])
    else:  # single_dose
        phe_max = compute_contraction(df, l['phe_single'], l['phe_end'])
        add_mn( 'Phe_3uM_mN',    phe_max)
        add_pct('Phe_3uM_%_KCl', phe_max / kcl60 * 100)
        phe_pct = None

    # Submaximal Phe (before ACh and SNP)
    phe_sub_Ach = compute_contraction(df, l['subPhe'],  l['subPhe_end'])
    phe_sub_SNP = compute_contraction(df, l['subPhe2'], l['subPhe2_end'])
    add_mn('subPhe_ACh_mN', phe_sub_Ach)
    add_mn('subPhe_SNP_mN', phe_sub_SNP)

    # ACh relaxation
    ach_abs = compute_cumulative_relaxation(df, ach_labels, l['ach_end'])
    ach_pct = ach_abs.div(phe_sub_Ach, axis=1) * 100
    for label in ach_labels:
        add_mn( f'ACh_{label}_mN', ach_abs.loc[label])
        add_pct(f'ACh_{label}_%',  ach_pct.loc[label])

    # SNP relaxation
    snp_abs = compute_cumulative_relaxation(df, snp_labels, l['snp_end'])
    snp_pct = snp_abs.div(phe_sub_SNP, axis=1) * 100
    for label in snp_labels:
        add_mn( f'SNP_{label}_mN', snp_abs.loc[label])
        add_pct(f'SNP_{label}_%',  snp_pct.loc[label])

    # mN rows first, then % rows
    result_df = pd.DataFrame(rows_mn + rows_pct).set_index('metric')

    # Store subtables for plotting (attach as attributes)
    result_df.attrs['kcl60']      = kcl60
    result_df.attrs['ach_%']      = ach_pct
    result_df.attrs['snp_%']      = snp_pct
    result_df.attrs['phe_%']      = phe_pct
    result_df.attrs['phe_labels'] = phe_labels if phe_mode == 'dose_response' else None

    return result_df

print('Functions loaded')

---
## Step 1 — Upload LabChart txt file

In [ ]:
def load_labchart(file_bytes):
    df = pd.read_csv(
        io.BytesIO(file_bytes), sep='\t', decimal=',', skiprows=9,
        header=None, names=['t',1,2,3,4,5,6,7,8,'comment'],
        dtype={c: float for c in ['t']+CHANNEL_COLS}, keep_default_na=False,

    )
    df['comment'] = df['comment'].astype(str).str.strip()
    for c in CHANNEL_COLS:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

uploaded   = files.upload()
filename   = list(uploaded.keys())[0]
df = load_labchart(uploaded[filename])
print(f'Loaded: {filename}')

---
## Step 2 — Preview protocol comments

In [ ]:
print(preview_comments(df).to_string(index=False))

---
## Step 3 — Protocol settings: labels
1) **`PHE_MODE`** — choose phenylephrine protocol:
- `'dose_response'` — cumulative concentrations (0.01–3 μM), full dose-response curve
- `'single_dose'`   — one dose only (3 μM), maximum contraction

2) Edit labels below only if your file uses different comments.

In [ ]:
# Choose Phe protocol - multiple doses or one!
PHE_MODE = 'single_dose'   # 'dose_response' or 'single_dose'

# ── Cumulative dose labels and concentrations ─────────────────────────────────
PHE_LABELS = ['Phe0,01', 'Phe0,03', 'Phe0,1', 'Phe0,3', 'Phe1', 'Phe3']
PHE_CONC   = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]   # uM

ACH_LABELS = ['Ach0,001', 'Ach0,01', 'Ach0,1', 'Ach1', 'Ach10']
ACH_CONC   = [0.001, 0.01, 0.1, 1.0, 10.0]       # uM

SNP_LABELS = ['SNP0,001', 'SNP0,01', 'SNP0,1', 'SNP1']
SNP_CONC   = [0.001, 0.01, 0.1, 1.0]              # uM

# Default protocol labels — edit if your file uses different annotations (preview above)
L_KCL60       = 'KCl60'    # KCl 60 mM addition
L_KCL60_END   = 'P2'       # rinse after KCl60
L_PHE_END     = 'PP'       # rinse after Phe: dose-response or maximal contraction
L_PHE_SINGLE  = 'Phe3uM'  # single-dose Phe label (maximal contraction)
L_SUBPHE      = 'subPhe'   # submaximal Phe precontraction before ACh
L_SUBPHE_END  = 'Ach0,001' # first ACh dose (= end of Phe precontraction window)
L_SUBPHE2     = '2subPhe'  # submaximal Phe precontraction before SNP
L_SUBPHE2_END = 'SNP0,001' # first SNP dose (= end of Phe precontraction window)
L_ACH_END     = 'P3'       # rinse after ACh
L_SNP_END     = 'K'        # end of recording

print(f'Phe mode: {PHE_MODE}')

---
## Step 4 — Calculate results

In [ ]:
# Build labels from protocol settings
LABELS = {
    'KCl60':       L_KCL60,
    'KCl60_end':   L_KCL60_END,
    'phe_end':     L_PHE_END,
    'phe_single':  L_PHE_SINGLE,
    'subPhe':      L_SUBPHE,
    'subPhe_end':  L_SUBPHE_END,
    'subPhe2':     L_SUBPHE2,
    'subPhe2_end': L_SUBPHE2_END,
    'ach_end':     L_ACH_END,
    'snp_end':     L_SNP_END,
}

results = analyse_experiment(df, PHE_MODE, PHE_LABELS, ACH_LABELS, SNP_LABELS, LABELS)

print(f'Results table: {results.shape[0]} metrics x {results.shape[1]} channels\n')
print(results.to_string())

---
## Step 5 — Summary plot

All 8 channels curves - quick check of the results.
If `PHE_MODE = 'single_dose'`, the Phe panel shows a bar chart of max contraction instead of a curve.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
ax_phe, ax_ach, ax_snp, ax_kcl = axes[0,0], axes[0,1], axes[1,0], axes[1,1]

# ── Phe panel ────────────────────────────────────────────────────────────────
if results.attrs['phe_%'] is not None:
    phe_pct = results.attrs['phe_%']
    for i, ch in enumerate(range(1, 9)):
        ax_phe.plot(PHE_CONC, phe_pct[ch].values, 'o-',
                    color=CH_COLORS[i], label=f'Ch {ch}',
                    markerfacecolor='white', markeredgewidth=1.5, markersize=5)
    ax_phe.set_xscale('log')
    ax_phe.set_xlabel('Phenylephrine (uM)')
    ax_phe.set_ylabel('Contraction (% KCl 60 mM)')
    ax_phe.set_title('Phenylephrine dose-response')
    ax_phe.set_ylim(bottom=0)
    ax_phe.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
    ax_phe.legend(fontsize=8, ncol=2, frameon=False)
else:
    # Single dose — bar chart
    phe_max_pct = results.loc['Phe_3uM_%_KCl']
    bars = ax_phe.bar([f'Ch {ch}' for ch in range(1, 9)], phe_max_pct.values,
                      color=CH_COLORS, edgecolor='white', width=0.65)
    for bar, val in zip(bars, phe_max_pct.values):
        ax_phe.text(bar.get_x()+bar.get_width()/2, val+0.5, f'{val:.1f}',
                    ha='center', va='bottom', fontsize=8)
    ax_phe.set_ylabel('Contraction (% KCl 60 mM)')
    ax_phe.set_title('Phenylephrine 3 uM — max contraction')
    ax_phe.legend(frameon=False)
    plt.setp(ax_phe.get_xticklabels(), rotation=30, ha='right')

# ── ACh relaxation ────────────────────────────────────────────────────────────
ach_pct = results.attrs['ach_%']
for i, ch in enumerate(range(1, 9)):
    ax_ach.plot(ACH_CONC, ach_pct[ch].values, 'o-',
                color=CH_COLORS[i], label=f'Ch {ch}',
                markerfacecolor='white', markeredgewidth=1.5, markersize=5)
ax_ach.set_xscale('log')
ax_ach.set_xlabel('Acetylcholine (uM)')
ax_ach.set_ylabel('Relaxation (% Phe pre-contraction)')
ax_ach.set_title('Endothelium-dependent relaxation (ACh)')
ax_ach.set_ylim(bottom=-20)
ax_ach.invert_yaxis()
ax_ach.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
ax_ach.legend(fontsize=8, ncol=2, frameon=False, loc='lower left')

# ── SNP relaxation ────────────────────────────────────────────────────────────
snp_pct = results.attrs['snp_%']
for i, ch in enumerate(range(1, 9)):
    ax_snp.plot(SNP_CONC, snp_pct[ch].values, 'o-',
                color=CH_COLORS[i], label=f'Ch {ch}',
                markerfacecolor='white', markeredgewidth=1.5, markersize=5)
ax_snp.set_xscale('log')
ax_snp.set_xlabel('SNP (uM)')
ax_snp.set_ylabel('Relaxation (% Phe pre-contraction)')
ax_snp.set_title('Endothelium-independent relaxation (SNP)')
ax_snp.set_ylim(bottom=-20)
ax_snp.invert_yaxis()
ax_snp.xaxis.set_major_formatter(ticker.LogFormatterSciNotation())
ax_snp.legend(fontsize=8, ncol=2, frameon=False, loc='lower left')

# ── KCl 60 mM barplot ─────────────────────────────────────────────────────────
kcl = results.attrs['kcl60']
bars = ax_kcl.bar([f'Ch {ch}' for ch in range(1, 9)], kcl.values,
                  color=CH_COLORS, edgecolor='white', width=0.65)
for bar, val in zip(bars, kcl.values):
    ax_kcl.text(bar.get_x()+bar.get_width()/2, val+0.04, f'{val:.2f}',
                ha='center', va='bottom', fontsize=8)
ax_kcl.set_ylabel('Contraction (mN)')
ax_kcl.set_title('KCl 60 mM — viability check')
ax_kcl.legend(frameon=False)
plt.setp(ax_kcl.get_xticklabels(), rotation=30, ha='right')

fig.suptitle(f'Wire Myography — {filename}  [{PHE_MODE}]',
             fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig('myograph_results.png', bbox_inches='tight', dpi=150)
plt.show()

---
## Step 6 — Download results

In [ ]:
output_name = filename.replace('.txt', '_results.xlsx')

# Export: single sheet, metric as index, ch1-ch8 as columns
with pd.ExcelWriter(output_name, engine='openpyxl') as writer:
    results.to_excel(writer, sheet_name='results')

files.download(output_name)
files.download('myograph_results.png')
print('Done')